# 11 · XAI cấp độ 1 trên Visual Genome: giải thích xếp hạng của CLIP + EdgeAwareGAT

Đề, mục 5 (cấp 1) và mục 7 (Nhiệm vụ 3): hiển thị quan hệ hoặc đường đi được dùng khi xếp hạng cho một số truy vấn,
phân tích 5 truy vấn thành công và 5 truy vấn thất bại; giải thích phải đến từ chính cơ chế xếp hạng (mục 2).
Notebook này còn đo fidelity, validity, sparsity theo cùng định nghĩa với notebook 10a, 10b để có dòng Visual Genome
trong bảng XAI chung.

**Mô hình được giải thích:** đúng các checkpoint đã khóa ở `results/VisualGenome/3SEED/seed_{2026,2027,2028}/best_gat.pt`,
α lấy từ checkpoint (chọn trên validation). Không huấn luyện lại, không chọn gì trên test.

**Bộ giải thích (giống notebook 06 của nhóm, áp dụng cho scene graph):** đơn vị là một quan hệ của ảnh, gồm cạnh gốc và
cạnh ngược (cạnh ảnh → thực thể `contains` cũng là đơn vị). Đóng góp Δ\_u = S\_full − S\_(full \ u), trong đó S là điểm đồ thị
q'·z của ảnh, z được tính lại bằng chính 2 tầng GAT của checkpoint sau khi bỏ u. Giải thích S = tối đa K\* đơn vị có Δ dương
lớn nhất. Attention của GAT chỉ dùng làm đường cơ sở: bộ mã hóa đồ thị không nhận truy vấn nên attention giống nhau cho mọi truy vấn.

| Bước | Nội dung | File kết quả (`0. final/results/VisualGenome/XAI/`) |
|---|---|---|
| V1 | Tái lập MRR test của seed 2026 từ checkpoint, đối chiếu `final_test_summary.json` | `V1_replay_check.json` |
| V2 | Chọn K\* trên **validation** (K nhỏ nhất có trung vị D(K)/D\_all ≥ 0,8) | `V2_Kstar_val.csv` |
| V3 | Test, 3 seed: giải thích ảnh hạng 1 trả về; bỏ / chỉ giữ / làm nhiễu S; so với K ngẫu nhiên và K theo attention | `V3_fidelity_per_query.csv`, `V3_fidelity_summary.csv`, `V3_tests.csv` |
| V4 | Đường cong theo K (seed chính) | `V4_sparsity_K_curve.csv` |
| V5 | Validity: phản thực (PN), vùng tiếp nhận, khớp nội dung truy vấn, ổn định giữa seed | `V5_validity.csv` |
| V6 | 5 thành công + 5 thất bại, có hình | `V6_cases.md`, `cases/*.png` |
| V7 | Demo: nhập câu truy vấn → top-k ảnh + đồ thị giải thích | `demo_*.png` |
| — | Dòng cho bảng XAI chung | `XAI_L3_row.csv` |

**Định nghĩa** (p = softmax của điểm cuối α·z(CLIP) + (1−α)·z(đồ thị) trên 50 ứng viên CLIP của truy vấn; ảnh được giải thích là
ảnh hạng 1 hệ thống trả về; F = mọi đơn vị trong đồ thị của ảnh):

| Chỉ số | Cách tính | Hướng tốt |
|---|---|---|
| Fid+ (cần thiết) | (p − p₋ₛ) / p khi bỏ S | ↑ |
| Fid− (đủ) | (p − pₛ) / p khi chỉ giữ S | ↓ |
| Characterization | 2·Fid+·(1 − Fid−) / (Fid+ + 1 − Fid−), sau khi kẹp Fid± về [0, 1] | ↑ |
| Fid làm nhiễu | (p − p_pert) / p khi thay đầu xa nút ảnh của mỗi cạnh trong S bằng một thực thể ngẫu nhiên khác của ảnh | ↑ |
| PN | % truy vấn mà bỏ S làm ảnh mất hạng 1 | ↑ |
| PS | % truy vấn mà chỉ giữ S ảnh vẫn hạng 1 | ↑ |
| Sparsity | 1 − \|S\| / \|F\| | ↑ (khi fidelity vẫn đạt) |
| Validity cấu trúc | (i) S chỉ gồm quan hệ của chính scene graph của ảnh; (ii) % đơn vị nằm trong vùng tiếp nhận 2 tầng | (i) = 100%, (ii) = 100% |
| Khớp nội dung | % đơn vị của S chạm thực thể có từ xuất hiện trong câu truy vấn, so với K ngẫu nhiên và tỉ lệ nền | ↑ |
| Ổn định | Jaccard của S giữa các seed khi ba seed trả về cùng ảnh | ↑ |

Fid± theo khảo sát XAI cho GNN (Amara và cs., arXiv 2207.12599). Đường cơ sở: K đơn vị ngẫu nhiên (10 lần) và K đơn vị
có attention lớn nhất.

**Chạy:** Colab GPU (T4 đủ) → Run all. Cần lối tắt `MLGs - Project` trong My Drive. V1 chạy toàn bộ 523.347 truy vấn test
(vài chục phút); đặt `RUN_FULL_REPLAY = False` để bỏ qua. Kết quả từng seed ở V3 được ghi ngay, phiên bị ngắt thì Run all lại.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'torch-geometric', 'pyarrow', 'pandas', 'scipy', 'networkx', 'open_clip_torch', 'requests'])
print('Đã cài thư viện')

## V0. Tìm `0. final` trên Drive, cấu hình

In [ ]:
import os, json, glob, time, random, math, hashlib
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.parquet as pq
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import GATConv

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Không chạy trên Colab:', exc)

def find_final_root():
    env = os.environ.get('FINAL_ROOT')
    if env and Path(env).is_dir():
        return Path(env)
    pats = ['/content/drive/MyDrive/MLGs - Project/0. final',
            '/content/drive/MyDrive/*/MLGs - Project/0. final',
            '/content/drive/MyDrive/*/*/MLGs - Project/0. final',
            '/content/drive/Shareddrives/*/MLGs - Project/0. final']
    for p in pats:
        hits = sorted(glob.glob(p))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError('Không tìm thấy "MLGs - Project/0. final". Thêm lối tắt vào My Drive hoặc đặt FINAL_ROOT.')

FINAL = find_final_root()
DATA_ROOT = FINAL / 'data/VisualGenome/processed'
RESULTS_ROOT = FINAL / 'results/VisualGenome'
GRAPH_ROOT, CLIP_ROOT = DATA_ROOT / 'graph', DATA_ROOT / 'clip'
FEATURE_ROOT = CLIP_ROOT / 'features'
OUT = RESULTS_ROOT / 'XAI'
(OUT / 'cases').mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MAIN_SEED = 2026
SEEDS = [2026, 2027, 2028]
SELECTION_SEED = 2026          # bốc truy vấn cố định, không nhìn kết quả
N_VAL_K = 300                  # số truy vấn val để chọn K*
N_TEST = 500                   # số truy vấn test đo fidelity mỗi seed
N_RANDOM = 10                  # số lần bỏ K đơn vị ngẫu nhiên
K_GRID = [1, 2, 3, 5, 8, 12, 16]
N_CASE_POOL = 20000            # số truy vấn test dùng để chọn ca 5 + 5
RUN_FULL_REPLAY = True
print('FINAL =', FINAL, '| DEVICE =', DEVICE)

## Nạp đồ thị, đặc trưng, tên thực thể và tên quan hệ (giống `evaluate_test.ipynb`)

In [ ]:
compact = torch.load(DATA_ROOT / 'graph_cache/compact_graph_cache_v2.pt', map_location='cpu', weights_only=False)
assert compact['node_ptr'][-1].item() == 2_609_364 and compact['edge_ptr'][-1].item() == 8_005_044
image_features = torch.load(FEATURE_ROOT / 'image_features.pt', map_location='cpu', weights_only=True).float()
entity_features = torch.load(FEATURE_ROOT / 'entity_label_features.pt', map_location='cpu', weights_only=True).float()
relation_features = torch.load(FEATURE_ROOT / 'relation_text_features.pt', map_location='cpu', weights_only=True).float()
relation_mapping = pd.read_parquet(GRAPH_ROOT / 'relation_mapping.parquet')
relation_feature_index = pd.read_parquet(FEATURE_ROOT / 'relation_feature_index.parquet')
relation_join = relation_mapping.merge(relation_feature_index[['edge_family', 'relation', 'feature_row']],
                                       on=['edge_family', 'relation'], how='left', validate='one_to_one')
relation_type_features = torch.empty((int(relation_mapping.relation_index.max()) + 1, 512), dtype=torch.float32)
known = relation_join.feature_row.notna().to_numpy()
relation_type_features[torch.from_numpy(relation_join.loc[known, 'relation_index'].to_numpy(np.int64))] = \
    relation_features[torch.from_numpy(relation_join.loc[known, 'feature_row'].to_numpy(np.int64))]
unk_rows = relation_feature_index.loc[relation_feature_index.learnable_relation_index == 1, 'feature_row'].to_numpy(np.int64)
unk_feature = F.normalize(relation_features[torch.from_numpy(unk_rows)].mean(0), dim=0)
missing = relation_join.loc[~known, 'relation_index'].to_numpy(np.int64)
assert set(missing).issubset({1})
if len(missing):
    relation_type_features[torch.from_numpy(missing)] = unk_feature
REL_NAME = dict(zip(relation_mapping.relation_index, relation_mapping.relation))

image_index = pd.read_parquet(FEATURE_ROOT / 'image_feature_index.parquet', columns=['image_id', 'feature_row'])
image_to_component = dict(zip(image_index.image_id.astype(str), image_index.feature_row))

def pick_text_col(df, prefer):
    cols = [c for c in df.columns
            if pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_object_dtype(df[c])]
    assert cols, f'Không có cột chữ trong {list(df.columns)}'
    pref = [c for c in cols if any(k in c.lower() for k in prefer)]
    return (pref or cols)[0]

efi = pd.read_parquet(FEATURE_ROOT / 'entity_feature_index.parquet')
print('entity_feature_index:', list(efi.columns))
ENT_LABEL_COL = pick_text_col(efi, ('label', 'name', 'canonical', 'text'))
ENT_LABEL = (dict(zip(efi.feature_row, efi[ENT_LABEL_COL])) if 'feature_row' in efi.columns
             else dict(enumerate(efi[ENT_LABEL_COL])))
print('Dùng cột tên thực thể:', ENT_LABEL_COL, '| ví dụ:', list(ENT_LABEL.values())[:8])

def node_label(g, v):
    return 'ẢNH' if v == g.root else str(ENT_LABEL.get(int(g.feature_rows[v]), f'thực thể {v}'))

def load_queries(split):
    q = pd.read_parquet(CLIP_ROOT / split / 'queries.parquet')
    col = pick_text_col(q.drop(columns=[c for c in q.columns if c.endswith('_id')], errors='ignore'),
                        ('text', 'phrase', 'caption', 'query', 'description'))
    return dict(zip(q.text_id.astype(str), q[col].astype(str)))

IMG_URL = {}
try:
    imgs = pd.read_parquet(DATA_ROOT / 'splits/images.parquet')
    ucol = [c for c in imgs.columns if 'url' in c.lower()]
    if ucol:
        IMG_URL = dict(zip(imgs.image_id.astype(str), imgs[ucol[0]].astype(str)))
    print('images.parquet:', list(imgs.columns), '| có URL ảnh:', bool(ucol))
except Exception as exc:
    print('Không đọc được URL ảnh (chỉ vẽ đồ thị):', exc)
print('Đã nạp đặc trưng')

## Mô hình và checkpoint đã khóa

In [ ]:
class EdgeAwareGAT(nn.Module):
    # Giữ nguyên kiến trúc của run_gat.py / evaluate_test.ipynb
    def __init__(self, input_dim=512, hidden_dim=128, heads=4, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.node_projection = nn.Linear(input_dim, hidden_dim)
        self.query_projection = nn.Linear(input_dim, hidden_dim)
        self.edge_projection = nn.Linear(input_dim + 1, hidden_dim)
        self.type_embedding = nn.Embedding(2, hidden_dim)
        self.gat1 = GATConv(hidden_dim, hidden_dim // heads, heads=heads, edge_dim=hidden_dim, dropout=dropout, add_self_loops=False)
        self.gat2 = GATConv(hidden_dim, hidden_dim // heads, heads=heads, edge_dim=hidden_dim, dropout=dropout, add_self_loops=False)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.logit_scale = nn.Parameter(torch.tensor(2.6592))

    def encode_graph(self, batch):
        h = self.node_projection(batch['x']) + self.type_embedding(batch['node_type'])
        edge_attr = self.edge_projection(torch.cat([batch['edge_attr'], batch['edge_reverse']], dim=1))
        update = self.gat1(h, batch['edge_index'], edge_attr=edge_attr)
        h = self.norm1(h + F.dropout(F.elu(update), self.dropout, self.training))
        update = self.gat2(h, batch['edge_index'], edge_attr=edge_attr)
        h = self.norm2(h + F.dropout(F.elu(update), self.dropout, self.training))
        return F.normalize(h[batch['root_local']], dim=1)

    def encode_query(self, q):
        return F.normalize(self.query_projection(q), dim=1)

RUNS = {
    'seed_2026': ('3SEED/seed_2026', 'real_edges'),
    'seed_2027': ('3SEED/seed_2027', 'real_edges'),
    'seed_2028': ('3SEED/seed_2028', 'real_edges'),
    'no_edge': ('ABLATION/no_edge_seed_2026', 'no_edge'),
    'random_edges': ('ABLATION/random_edges_seed_2026', 'random_edges'),
}
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()

MODELS = {}
for name, (rel, mode) in RUNS.items():
    path = RESULTS_ROOT / rel / 'best_gat.pt'
    ck = torch.load(path, map_location='cpu', weights_only=False)
    assert ck.get('edge_mode', 'real_edges') == mode, (name, ck.get('edge_mode'))
    m = EdgeAwareGAT().to(DEVICE)
    m.load_state_dict(ck['model_state_dict'])
    m.eval()
    MODELS[name] = {'model': m, 'alpha': float(ck['alpha']), 'epoch': int(ck['epoch']), 'edge_mode': mode,
                    'sha256': sha256(path)}
    print(f"{name}: epoch {ck['epoch']}, α_CLIP = {ck['alpha']}, edge_mode = {mode}")

## Lõi bộ giải thích (nhúng nguyên văn `vg_xai_core.py`, đã kiểm thử khớp mã gốc trên dữ liệu giả lập)

In [ ]:
# Lõi bộ giải thích cấp 1 cho CLIP + EdgeAwareGAT trên Visual Genome.
# Được nhúng nguyên văn vào notebook 11_visualgenome_gat_xai_level1.ipynb (ô "Lõi bộ giải thích").
#
# Đơn vị giải thích = một quan hệ của scene graph trong ảnh: cạnh gốc và cạnh ngược của nó
# (cùng hai đầu, cùng loại quan hệ) gộp thành một đơn vị. Cạnh ảnh -> thực thể ("contains") cũng là đơn vị.
# Đóng góp của đơn vị u: delta_u = S_full - S_(full \ u), trong đó S là điểm đồ thị q'.z của ảnh và
# z được tính lại bằng chính 2 tầng GAT của checkpoint sau khi bỏ u (giống bộ giải thích của notebook 06).
import math
import random

import numpy as np
import torch
import torch.nn.functional as F


class ComponentGraph:
    """Đồ thị của một ảnh (một thành phần liên thông trong compact cache), đặt sẵn trên thiết bị."""

    def __init__(self, compact, comp, image_features, entity_features, relation_type_features,
                 device, edge_mode="real_edges", random_edge_seed=2026):
        ns, ne = compact["node_ptr"][comp:comp + 2].tolist()
        es, ee = compact["edge_ptr"][comp:comp + 2].tolist()
        nodes = compact["node_perm"][ns:ne]
        edges = compact["edge_index"][:, es:ee].clone()
        if edge_mode == "random_edges":
            gen = torch.Generator().manual_seed(int(random_edge_seed) + int(comp))
            perm = torch.randperm(edges.shape[1], generator=gen)
            edges = torch.stack((edges[0], edges[1, perm]))
        g2l = {int(n): i for i, n in enumerate(nodes.tolist())}
        src = torch.tensor([g2l[int(v)] for v in edges[0].tolist()], dtype=torch.long)
        dst = torch.tensor([g2l[int(v)] for v in edges[1].tolist()], dtype=torch.long)
        node_type = compact["node_type"][nodes].long()
        roots = (node_type == 1).nonzero().flatten()
        assert roots.numel() == 1, "mỗi thành phần phải có đúng một nút ảnh"
        feat_rows = compact["node_feature_row"][nodes]
        x = torch.empty((nodes.numel(), image_features.shape[1]), dtype=torch.float32)
        img = node_type == 1
        x[img] = image_features[feat_rows[img]]
        x[~img] = entity_features[feat_rows[~img]]
        rel = compact["edge_relation_type"][es:ee].long()
        rev = compact["edge_is_reverse"][es:ee].float()

        self.comp = int(comp)
        self.device = device
        self.edge_mode = edge_mode
        self.nodes_global = nodes
        self.feature_rows = feat_rows
        self.node_type = node_type.to(device)
        self.x = x.to(device)
        self.root = int(roots.item())
        self.src, self.dst = src, dst
        self.rel = rel
        self.rev = rev
        self.edge_attr = relation_type_features[rel].to(device)
        self.edge_reverse = rev.unsqueeze(1).to(device)
        self.n_nodes = int(nodes.numel())
        self.n_edges = int(src.numel())
        # Gộp cạnh gốc và cạnh ngược thành đơn vị quan hệ.
        groups = {}
        for e in range(self.n_edges):
            a, b = int(src[e]), int(dst[e])
            key = (min(a, b), max(a, b), int(rel[e]))
            groups.setdefault(key, []).append(e)
        self.unit_keys = list(groups.keys())
        self.unit_edges = [groups[k] for k in self.unit_keys]
        self.n_units = len(self.unit_keys)
        um = torch.zeros((self.n_units, self.n_edges), dtype=torch.bool)
        for u, es_ in enumerate(self.unit_edges):
            um[u, es_] = True
        self.unit_edge_mask = um
        # Khoảng cách tới nút ảnh (vô hướng) để kiểm tra vùng tiếp nhận 2 tầng.
        adj = [[] for _ in range(self.n_nodes)]
        for a, b in zip(src.tolist(), dst.tolist()):
            adj[a].append(b)
            adj[b].append(a)
        dist = [math.inf] * self.n_nodes
        dist[self.root] = 0
        frontier = [self.root]
        while frontier:
            nxt = []
            for v in frontier:
                for w in adj[v]:
                    if dist[w] == math.inf:
                        dist[w] = dist[v] + 1
                        nxt.append(w)
            frontier = nxt
        self.dist = dist
        self.unit_is_contains = [k[2] == 0 for k in self.unit_keys]

    def unit_in_receptive_field(self, u):
        a, b, _ = self.unit_keys[u]
        return min(self.dist[a], self.dist[b]) <= 1

    def keep_mask_without(self, units):
        keep = torch.ones(self.n_edges, dtype=torch.bool)
        if len(units):
            keep &= ~self.unit_edge_mask[list(units)].any(0)
        return keep

    def keep_mask_only(self, units):
        if len(units) == 0:
            return torch.zeros(self.n_edges, dtype=torch.bool)
        return self.unit_edge_mask[list(units)].any(0)


def _encode_batched(model, g, masks, edge_index_override=None):
    """Mã hóa nút ảnh cho nhiều biến thể của cùng một đồ thị trong một lần chạy.

    masks: bool [M, E] (cạnh được giữ). edge_index_override: list độ dài M gồm (src, dst) thay thế hoặc None.
    """
    M = masks.shape[0]
    n = g.n_nodes
    xs, types, eis, eas, ers, roots = [], [], [], [], [], []
    for m in range(M):
        keep = masks[m]
        if edge_index_override is not None and edge_index_override[m] is not None:
            s, d = edge_index_override[m]
        else:
            s, d = g.src, g.dst
        ei = torch.stack((s[keep], d[keep])) + m * n
        eis.append(ei)
        eas.append(g.edge_attr[keep.to(g.device)])
        ers.append(g.edge_reverse[keep.to(g.device)])
        roots.append(g.root + m * n)
    x = g.x.repeat(M, 1)
    nt = g.node_type.repeat(M)
    batch = {
        "x": x,
        "node_type": nt,
        "edge_index": torch.cat(eis, 1).to(g.device),
        "edge_attr": torch.cat(eas, 0),
        "edge_reverse": torch.cat(ers, 0),
        "root_local": torch.tensor(roots, device=g.device),
    }
    if g.edge_mode == "no_edge":
        batch["edge_index"] = batch["edge_index"][:, :0]
        batch["edge_attr"] = batch["edge_attr"][:0]
        batch["edge_reverse"] = batch["edge_reverse"][:0]
    return model.encode_graph(batch)


@torch.no_grad()
def graph_scores(model, g, qz, masks, edge_index_override=None, chunk=64):
    """Điểm đồ thị q'.z của ảnh cho từng biến thể (mask)."""
    out = []
    for i in range(0, masks.shape[0], chunk):
        ov = None if edge_index_override is None else edge_index_override[i:i + chunk]
        z = _encode_batched(model, g, masks[i:i + chunk], ov)
        out.append(z @ qz)
    return torch.cat(out).float().cpu()


@torch.no_grad()
def unit_deltas(model, g, qz):
    """delta_u = S_full - S_(full \\ u) cho mọi đơn vị; trả về (S_full, S_none, delta[U])."""
    full = torch.ones((1, g.n_edges), dtype=torch.bool)
    none = torch.zeros((1, g.n_edges), dtype=torch.bool)
    leave = ~g.unit_edge_mask  # [U, E]: bỏ đúng một đơn vị
    s = graph_scores(model, g, qz, torch.cat([full, none, leave], 0))
    return float(s[0]), float(s[1]), (s[0] - s[2:])


def explanation(deltas, K):
    """S = tối đa K đơn vị có delta dương lớn nhất."""
    order = torch.argsort(deltas, descending=True).tolist()
    return [u for u in order[:K] if deltas[u] > 0]


@torch.no_grad()
def attention_unit_scores(model, g):
    """Điểm attention của từng đơn vị (tổng attention trung bình các head, cả hai tầng). Chỉ làm đường cơ sở."""
    ei = torch.stack((g.src, g.dst)).to(g.device)
    h = model.node_projection(g.x) + model.type_embedding(g.node_type)
    ea = model.edge_projection(torch.cat([g.edge_attr, g.edge_reverse], 1))
    upd, (ei1, a1) = model.gat1(h, ei, edge_attr=ea, return_attention_weights=True)
    h = model.norm1(h + F.elu(upd))
    _, (ei2, a2) = model.gat2(h, ei, edge_attr=ea, return_attention_weights=True)
    # GATConv không tự thêm self-loop (add_self_loops=False) nên thứ tự cạnh giữ nguyên.
    assert ei1.shape[1] == g.n_edges and ei2.shape[1] == g.n_edges
    per_edge = (a1.mean(1) + a2.mean(1)).cpu()
    return torch.stack([per_edge[e].sum() for e in g.unit_edges])


def rewired_edges(g, units, rng):
    """Làm nhiễu S: thay đầu không phải nút ảnh của mỗi cạnh trong S bằng một thực thể ngẫu nhiên khác của ảnh."""
    s, d = g.src.clone(), g.dst.clone()
    entities = [i for i in range(g.n_nodes) if i != g.root]
    for u in units:
        a, b, _ = g.unit_keys[u]
        for e in g.unit_edges[u]:
            # giữ đầu gần nút ảnh hơn, thay đầu còn lại
            if g.dist[int(s[e])] <= g.dist[int(d[e])]:
                cur = int(d[e])
                pool = [v for v in entities if v != cur and v != int(s[e])]
                if pool:
                    d[e] = rng.choice(pool)
            else:
                cur = int(s[e])
                pool = [v for v in entities if v != cur and v != int(d[e])]
                if pool:
                    s[e] = rng.choice(pool)
    return s, d


def fuse(clip_scores, graph_scores_vec, alpha):
    cz = (clip_scores - clip_scores.mean()) / clip_scores.std().clamp_min(1e-6)
    gz = (graph_scores_vec - graph_scores_vec.mean()) / graph_scores_vec.std().clamp_min(1e-6)
    return alpha * cz + (1.0 - alpha) * gz


def prob_rank(clip_scores, graph_vec, alpha, i, new_graph_score=None):
    """Xác suất softmax và hạng của ứng viên i khi điểm đồ thị của i được thay bằng new_graph_score."""
    gv = graph_vec.clone()
    if new_graph_score is not None:
        gv[i] = new_graph_score
    fused = fuse(clip_scores, gv, alpha)
    p = torch.softmax(fused, 0)[i].item()
    rank = 1 + int((fused > fused[i]).sum())
    return p, rank


def characterization(fid_plus, fid_minus):
    a = min(max(fid_plus, 0.0), 1.0)
    b = 1.0 - min(max(fid_minus, 0.0), 1.0)
    return 0.0 if a + b == 0 else 2 * a * b / (a + b)


def tokens(text):
    import re
    return set(w for w in re.findall(r"[a-z]+", str(text).lower()) if len(w) > 2)


def unit_grounded(g, u, query_tokens, node_label):
    """Đơn vị chạm một thực thể có từ xuất hiện trong câu truy vấn (validity theo nội dung)."""
    a, b, _ = g.unit_keys[u]
    for v in (a, b):
        if v == g.root:
            continue
        if tokens(node_label(g, v)) & query_tokens:
            return True
    return False


@torch.no_grad()
def explain_and_measure(model, g, qz, clip_scores, graph_vec, alpha, i, K, n_random=10, seed=2026,
                        query_tokens=None, node_label=None):
    """Đo đủ bộ chỉ số cho giải thích của ứng viên i (ảnh hạng 1 trả về)."""
    rng = random.Random(seed)
    s_full, s_none, deltas = unit_deltas(model, g, qz)
    S = explanation(deltas, K)
    U = g.n_units
    k = len(S)
    p0, r0 = prob_rank(clip_scores, graph_vec, alpha, i)
    out = {"S_full": s_full, "S_none": s_none, "n_units": U, "size_S": k, "p0": p0, "rank0": r0,
           "S": S, "deltas_S": [float(deltas[u]) for u in S]}
    if k == 0:
        out["explained"] = False
        return out
    out["explained"] = True
    masks = [g.keep_mask_without(S), g.keep_mask_only(S)]
    rand_sets = [rng.sample(range(U), k) for _ in range(n_random)]
    masks += [g.keep_mask_without(R) for R in rand_sets]
    masks += [g.keep_mask_only(R) for R in rand_sets]
    att = attention_unit_scores(model, g)
    A = torch.argsort(att, descending=True).tolist()[:k]
    masks += [g.keep_mask_without(A)]
    sc = graph_scores(model, g, qz, torch.stack(masks))
    s_rm, s_only = float(sc[0]), float(sc[1])
    s_rand_rm = sc[2:2 + n_random]
    s_rand_only = sc[2 + n_random:2 + 2 * n_random]
    s_att_rm = float(sc[-1])
    p_rm, r_rm = prob_rank(clip_scores, graph_vec, alpha, i, s_rm)
    p_only, r_only = prob_rank(clip_scores, graph_vec, alpha, i, s_only)
    p_att, r_att = prob_rank(clip_scores, graph_vec, alpha, i, s_att_rm)
    pr = [prob_rank(clip_scores, graph_vec, alpha, i, float(v)) for v in s_rand_rm]
    po = [prob_rank(clip_scores, graph_vec, alpha, i, float(v)) for v in s_rand_only]
    # làm nhiễu: thay đầu kia
    s_pert, d_pert = rewired_edges(g, S, rng)
    s_perturb = float(graph_scores(model, g, qz, torch.ones((1, g.n_edges), dtype=torch.bool),
                                   edge_index_override=[(s_pert, d_pert)])[0])
    p_pert, r_pert = prob_rank(clip_scores, graph_vec, alpha, i, s_perturb)
    fp = (p0 - p_rm) / p0
    fm = (p0 - p_only) / p0
    out.update({
        "fid_plus": fp,
        "fid_plus_rand": float(np.mean([(p0 - p) / p0 for p, _ in pr])),
        "fid_plus_attn": (p0 - p_att) / p0,
        "fid_minus": fm,
        "fid_minus_rand": float(np.mean([(p0 - p) / p0 for p, _ in po])),
        "characterization": characterization(fp, fm),
        "fid_perturb": (p0 - p_pert) / p0,
        "dscore_rm": s_full - s_rm,
        "dscore_rand": float(np.mean([s_full - float(v) for v in s_rand_rm])),
        "drank_rm": r_rm - r0,
        "drank_rand": float(np.mean([r - r0 for _, r in pr])),
        "drank_attn": r_att - r0,
        "PN": float(r_rm > 1),
        "PN_rand": float(np.mean([r > 1 for _, r in pr])),
        "PN_attn": float(r_att > 1),
        "PN_perturb": float(r_pert > 1),
        "PS": float(r_only == 1),
        "PS_rand": float(np.mean([r == 1 for _, r in po])),
        "sparsity": 1.0 - k / U,
        "in_receptive_field": float(np.mean([g.unit_in_receptive_field(u) for u in S])),
        "share_contains": float(np.mean([g.unit_is_contains[u] for u in S])),
        "win_rate": float(np.mean([(p0 - p_rm) > (p0 - p) for p, _ in pr])),
    })
    if query_tokens is not None and node_label is not None:
        out["ground_S"] = float(np.mean([unit_grounded(g, u, query_tokens, node_label) for u in S]))
        out["ground_rand"] = float(np.mean([np.mean([unit_grounded(g, u, query_tokens, node_label) for u in R])
                                            for R in rand_sets]))
        out["ground_base"] = float(np.mean([unit_grounded(g, u, query_tokens, node_label) for u in range(U)]))
    return out


@torch.no_grad()
def completeness_curve(model, g, qz, Ks):
    """D(K)/D_all: mức giảm điểm đồ thị khi bỏ top-K so với bỏ mọi đơn vị có delta dương (để chọn K* trên val)."""
    s_full, _, deltas = unit_deltas(model, g, qz)
    pos = [u for u in torch.argsort(deltas, descending=True).tolist() if deltas[u] > 0]
    if not pos:
        return None
    masks = [g.keep_mask_without(pos)] + [g.keep_mask_without(pos[:K]) for K in Ks]
    sc = graph_scores(model, g, qz, torch.stack(masks))
    d_all = s_full - float(sc[0])
    if d_all <= 1e-9:
        return None
    return {K: (s_full - float(sc[1 + j])) / d_all for j, K in enumerate(Ks)}

## Chấm điểm trên 50 ứng viên CLIP của mỗi truy vấn (giống `evaluate_frozen_test`)

In [ ]:
global_to_local = torch.full((2_609_364,), -1, dtype=torch.long)

def build_component_batch(component_rows, edge_mode):
    node_chunks, edge_chunks, rel_chunks, rev_chunks, roots = [], [], [], [], []
    for comp in component_rows:
        ns, ne = compact['node_ptr'][comp:comp + 2].tolist()
        es, ee = compact['edge_ptr'][comp:comp + 2].tolist()
        nodes = compact['node_perm'][ns:ne]
        edges = compact['edge_index'][:, es:ee]
        if edge_mode == 'random_edges':
            gen = torch.Generator().manual_seed(2026 + comp)
            perm = torch.randperm(edges.shape[1], generator=gen)
            edges = torch.stack((edges[0], edges[1, perm]))
        nt = compact['node_type'][nodes]
        roots.append(nodes[nt == 1].item())
        node_chunks.append(nodes); edge_chunks.append(edges)
        rel_chunks.append(compact['edge_relation_type'][es:ee]); rev_chunks.append(compact['edge_is_reverse'][es:ee])
    ng = torch.cat(node_chunks); eg = torch.cat(edge_chunks, 1)
    global_to_local[ng] = torch.arange(ng.numel())
    el = global_to_local[eg]; rl = global_to_local[torch.tensor(roots)]
    nt = compact['node_type'][ng].long(); fr = compact['node_feature_row'][ng]
    x = torch.empty((ng.numel(), 512)); im = nt == 1
    x[im] = image_features[fr[im]]; x[~im] = entity_features[fr[~im]]
    ea = relation_type_features[torch.cat(rel_chunks)]
    er = torch.cat(rev_chunks).float().unsqueeze(1)
    if edge_mode == 'no_edge':
        el, ea, er = el[:, :0], ea[:0], er[:0]
    global_to_local[ng] = -1
    return {'x': x.to(DEVICE), 'node_type': nt.to(DEVICE), 'edge_index': el.to(DEVICE),
            'edge_attr': ea.to(DEVICE), 'edge_reverse': er.to(DEVICE), 'root_local': rl.to(DEVICE)}

@torch.no_grad()
def encode_candidates(name, image_ids, bs=128):
    spec = MODELS[name]
    comps = [image_to_component[x] for x in image_ids]
    out = [spec['model'].encode_graph(build_component_batch(comps[i:i + bs], spec['edge_mode'])).cpu()
           for i in range(0, len(comps), bs)]
    return torch.cat(out)

SPLIT = {}
def load_split(split):
    root = CLIP_ROOT / split
    cand = pd.read_parquet(root / 'candidate_images.parquet', columns=['image_id']).image_id.astype(str).tolist()
    res = pq.read_table(root / 'retrieval_results.parquet',
                        columns=['text_id', 'best_positive_rank', 'topk_image_ids', 'topk_scores', 'positive_image_id']).to_pandas()
    ti = pd.read_parquet(root / 'text_embedding_index.parquet')
    trow = dict(zip(ti.text_id.astype(str), ti.embedding_row))
    tfeat = torch.load(root / 'text_embeddings.pt', map_location='cpu', weights_only=True)
    SPLIT[split] = {'cand': cand, 'row': {x: i for i, x in enumerate(cand)}, 'res': res, 'trow': trow,
                    'tfeat': tfeat, 'text': load_queries(split), 'emb': {}}
    print(split, ':', len(cand), 'ảnh,', len(res), 'truy vấn')

def split_emb(split, name):
    S = SPLIT[split]
    if name not in S['emb']:
        S['emb'][name] = encode_candidates(name, S['cand'])
    return S['emb'][name]

@torch.no_grad()
def score_row(split, name, row):
    # Trả về: ids (50), clip_scores, graph_vec, qz, fused, hạng ảnh đúng, chỉ số ảnh hạng 1
    S, spec = SPLIT[split], MODELS[name]
    q = S['tfeat'][S['trow'][str(row.text_id)]].float().unsqueeze(0).to(DEVICE)
    qz = spec['model'].encode_query(q)[0]
    ids = [str(x) for x in row.topk_image_ids]
    emb = split_emb(split, name)
    gv = (emb[[S['row'][x] for x in ids]] @ qz.cpu()).float()
    clip = torch.tensor(row.topk_scores, dtype=torch.float32)
    fused = fuse(clip, gv, spec['alpha'])
    pos = str(row.positive_image_id)
    rank = 1 + int((fused > fused[ids.index(pos)]).sum()) if pos in ids else int(row.best_positive_rank)
    clip_rank = (ids.index(pos) + 1) if pos in ids else int(row.best_positive_rank)
    return ids, clip, gv, qz, fused, rank, clip_rank, int(torch.argmax(fused))

load_split('val'); load_split('test')

## V1. Tái lập kết quả test của seed 2026 từ checkpoint

In [ ]:
summary = json.loads((RESULTS_ROOT / 'final_test_summary.json').read_text())
expected = summary['runs']['main_seed_2026']['test_metrics']['MRR']
if RUN_FULL_REPLAY:
    t0 = time.time(); rr = 0.0
    res = SPLIT['test']['res']
    for row in res.itertuples(index=False):
        rr += 1.0 / score_row('test', 'seed_2026', row)[5]
    got = rr / len(res)
    replay = {'MRR_replay': got, 'MRR_final_test_summary': expected, 'abs_diff': abs(got - expected),
              'n_queries': len(res), 'minutes': (time.time() - t0) / 60}
    print(replay)
    assert abs(got - expected) < 2e-4, 'Không tái lập được MRR test: dừng lại kiểm tra dữ liệu và checkpoint'
else:
    replay = {'skipped': True, 'MRR_final_test_summary': expected}
(OUT / 'V1_replay_check.json').write_text(json.dumps(replay, indent=2))

## V2. Chọn K\* trên validation

Giải thích ảnh hạng 1 mà hệ thống trả về cho `N_VAL_K` truy vấn val (seed chính). D(K) = mức giảm điểm đồ thị khi bỏ top-K đơn vị;
D\_all = mức giảm khi bỏ mọi đơn vị có Δ dương. K\* = K nhỏ nhất trong lưới có trung vị D(K)/D\_all ≥ 0,8 (cùng quy tắc với cấp 2, 3).

In [ ]:
rng = np.random.default_rng(SELECTION_SEED)
val_res = SPLIT['val']['res']
val_rows = val_res.iloc[np.sort(rng.choice(len(val_res), N_VAL_K, replace=False))]
curves = []
for row in val_rows.itertuples(index=False):
    ids, clip, gv, qz, fused, rank, crank, top = score_row('val', 'seed_2026', row)
    g = ComponentGraph(compact, image_to_component[ids[top]], image_features, entity_features, relation_type_features, DEVICE)
    c = completeness_curve(MODELS['seed_2026']['model'], g, qz, K_GRID)
    if c is not None:
        curves.append(c)
kdf = pd.DataFrame([{'K': K, 'median_completeness': float(np.median([c[K] for c in curves])),
                     'mean_completeness': float(np.mean([c[K] for c in curves])), 'n': len(curves)} for K in K_GRID])
K_STAR = int(next((r.K for r in kdf.itertuples() if r.median_completeness >= 0.8), K_GRID[-1]))
kdf['is_K_star'] = kdf.K == K_STAR
kdf.to_csv(OUT / 'V2_Kstar_val.csv', index=False)
print(kdf.to_string(index=False)); print('K* =', K_STAR)

## V3. Fidelity tại K\* trên test, 3 seed

Cùng `N_TEST` truy vấn test (bốc một lần với `SELECTION_SEED`) cho cả 3 seed. Mỗi truy vấn: giải thích ảnh hạng 1 mà seed đó trả về,
kiểm tra điểm tái tạo khớp điểm dùng khi xếp hạng, rồi đo mọi biến thể.

In [ ]:
test_res = SPLIT['test']['res']
test_rows = test_res.iloc[np.sort(np.random.default_rng(SELECTION_SEED).choice(len(test_res), N_TEST, replace=False))]
rows_out = []
for seed in SEEDS:
    name = f'seed_{seed}'
    cache = OUT / f'_V3_{name}.csv'
    if cache.exists():
        rows_out.append(pd.read_csv(cache)); print(name, ': dùng lại', cache.name); continue
    spec, recs, t0 = MODELS[name], [], time.time()
    for row in test_rows.itertuples(index=False):
        ids, clip, gv, qz, fused, rank, crank, top = score_row('test', name, row)
        g = ComponentGraph(compact, image_to_component[ids[top]], image_features, entity_features, relation_type_features, DEVICE)
        s_check = float(graph_scores(spec['model'], g, qz, torch.ones((1, g.n_edges), dtype=torch.bool))[0])
        r = explain_and_measure(spec['model'], g, qz, clip, gv, spec['alpha'], top, K_STAR, n_random=N_RANDOM,
                                seed=SELECTION_SEED, query_tokens=tokens(SPLIT['test']['text'].get(str(row.text_id), '')),
                                node_label=node_label)
        r.update({'seed': seed, 'text_id': str(row.text_id), 'returned_image': ids[top],
                  'returned_is_relevant': ids[top] == str(row.positive_image_id), 'rank_positive': rank,
                  'clip_rank_positive': crank, 'replay_abs_diff': abs(s_check - float(gv[top])),
                  'S_keys': json.dumps([list(g.unit_keys[u]) for u in r['S']])})
        r.pop('S'); r['deltas_S'] = json.dumps(r['deltas_S'])
        recs.append(r)
    df = pd.DataFrame(recs); df.to_csv(cache, index=False); rows_out.append(df)
    print(f"{name}: {len(df)} truy vấn, {df.explained.sum()} có giải thích, "
          f"sai khác điểm tái tạo lớn nhất {df.replay_abs_diff.max():.2e}, {(time.time()-t0)/60:.1f} phút")
per_query = pd.concat(rows_out, ignore_index=True)
assert per_query.replay_abs_diff.max() < 1e-4, 'Điểm tái tạo không khớp điểm xếp hạng'
per_query.to_csv(OUT / 'V3_fidelity_per_query.csv', index=False)

In [ ]:
from scipy.stats import wilcoxon
M = ['fid_plus', 'fid_plus_rand', 'fid_plus_attn', 'fid_minus', 'fid_minus_rand', 'characterization', 'fid_perturb',
     'dscore_rm', 'dscore_rand', 'drank_rm', 'drank_rand', 'drank_attn', 'PN', 'PN_rand', 'PN_attn', 'PN_perturb',
     'PS', 'PS_rand', 'sparsity', 'size_S', 'n_units', 'in_receptive_field', 'share_contains', 'win_rate',
     'ground_S', 'ground_rand', 'ground_base', 'returned_is_relevant']
ex = per_query[per_query.explained]
per_seed = ex.groupby('seed')[M].mean()
per_seed['n'] = ex.groupby('seed').size()
per_seed['fid_minus_median'] = ex.groupby('seed').fid_minus.median()
per_seed['fid_perturb_median'] = ex.groupby('seed').fid_perturb.median()
summ = pd.DataFrame({'mean': per_seed.mean(), 'std': per_seed.std(ddof=1)})
summ.to_csv(OUT / 'V3_fidelity_summary.csv'); per_seed.to_csv(OUT / 'V3_fidelity_per_seed.csv')
print(summ.round(4).to_string())

def boot_ci(d, n=5000, seed=0):
    r = np.random.default_rng(seed); d = np.asarray(d)
    m = [r.choice(d, len(d)).mean() for _ in range(n)]
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))
avg = ex.groupby('text_id')[M].mean()
tests = []
for a, b, label in [('fid_plus', 'fid_plus_rand', 'Fid+: S so với K ngẫu nhiên'), ('fid_plus', 'fid_plus_attn', 'Fid+: S so với K attention'),
                    ('PN', 'PN_rand', 'PN: S so với K ngẫu nhiên'), ('PS', 'PS_rand', 'PS: S so với K ngẫu nhiên'),
                    ('ground_S', 'ground_rand', 'Khớp nội dung: S so với K ngẫu nhiên')]:
    d = (avg[a] - avg[b]).dropna()
    lo, hi = boot_ci(d)
    p = wilcoxon(d, alternative='greater').pvalue if (d != 0).any() else float('nan')
    tests.append({'test': label, 'n': len(d), 'mean_a': avg[a].mean(), 'mean_b': avg[b].mean(), 'delta': d.mean(),
                  'ci95_low': lo, 'ci95_high': hi, 'wilcoxon_p_one_sided': p})
tests = pd.DataFrame(tests); tests.to_csv(OUT / 'V3_tests.csv', index=False)
print(tests.round(4).to_string(index=False))

## V4. Đường cong theo K (seed chính, cùng truy vấn test)

In [ ]:
spec, recs = MODELS['seed_2026'], []
for row in test_rows.itertuples(index=False):
    ids, clip, gv, qz, fused, rank, crank, top = score_row('test', 'seed_2026', row)
    g = ComponentGraph(compact, image_to_component[ids[top]], image_features, entity_features, relation_type_features, DEVICE)
    for K in K_GRID:
        r = explain_and_measure(spec['model'], g, qz, clip, gv, spec['alpha'], top, K, n_random=3, seed=SELECTION_SEED)
        if r['explained']:
            recs.append({'K': K, 'fid_plus': r['fid_plus'], 'fid_plus_rand': r['fid_plus_rand'], 'fid_minus': r['fid_minus'],
                         'PN': r['PN'], 'PS': r['PS'], 'sparsity': r['sparsity'], 'size_S': r['size_S']})
kc = pd.DataFrame(recs).groupby('K').mean().reset_index()
kc['is_K_star'] = kc.K == K_STAR
kc.to_csv(OUT / 'V4_sparsity_K_curve.csv', index=False); print(kc.round(4).to_string(index=False))

## V5. Validity và ổn định giữa seed

In [ ]:
val_rows_out = []
for seed, d in ex.groupby('seed'):
    val_rows_out.append({'seed': seed, 'n': len(d), 'PN': d.PN.mean(), 'PN_rand': d.PN_rand.mean(), 'PN_attn': d.PN_attn.mean(),
                         'PN_perturb': d.PN_perturb.mean(), 'PS': d.PS.mean(), 'PS_rand': d.PS_rand.mean(),
                         'in_own_scene_graph': 1.0, 'in_receptive_field': d.in_receptive_field.mean(),
                         'ground_S': d.ground_S.mean(), 'ground_rand': d.ground_rand.mean(), 'ground_base': d.ground_base.mean(),
                         'returned_is_relevant': d.returned_is_relevant.mean()})
# Ổn định: Jaccard của S giữa các cặp seed khi cùng trả về một ảnh
jac = []
for tid, d in ex.groupby('text_id'):
    if len(d) == 3 and d.returned_image.nunique() == 1:
        sets = [set(map(tuple, json.loads(s))) for s in d.S_keys]
        pairs = [(0, 1), (0, 2), (1, 2)]
        jac.append(np.mean([len(sets[a] & sets[b]) / max(1, len(sets[a] | sets[b])) for a, b in pairs]))
vdf = pd.DataFrame(val_rows_out)
vdf['stability_jaccard'] = float(np.mean(jac)) if jac else float('nan')
vdf['stability_n'] = len(jac)
vdf.to_csv(OUT / 'V5_validity.csv', index=False); print(vdf.round(4).to_string(index=False))

row = {'Cấp': 1, 'Dữ liệu': 'Visual Genome', 'Mô hình': 'CLIP + EdgeAwareGAT', 'Bộ giải thích': 'che quan hệ (occlusion)',
       'Số truy vấn mỗi seed': int(per_seed.n.mean()), 'K*': K_STAR}
for col, key in [('Fid+', 'fid_plus'), ('Fid+ ngẫu nhiên', 'fid_plus_rand'), ('Fid+ attention', 'fid_plus_attn'),
                 ('Characterization', 'characterization'), ('PN', 'PN'), ('PN ngẫu nhiên', 'PN_rand'), ('PS', 'PS'),
                 ('Sparsity', 'sparsity')]:
    row[col] = f"{summ.loc[key, 'mean']:.3f} ± {summ.loc[key, 'std']:.3f}"
row['Fid− (trung vị)'] = f"{summ.loc['fid_minus_median', 'mean']:.3f} ± {summ.loc['fid_minus_median', 'std']:.3f}"
row['Fid làm nhiễu (trung vị)'] = f"{summ.loc['fid_perturb_median', 'mean']:.3f} ± {summ.loc['fid_perturb_median', 'std']:.3f}"
row['Validity cấu trúc'] = f"{100 * vdf.in_receptive_field.mean():.0f}%"
row['Ổn định giữa seed (Jaccard)'] = f"{vdf.stability_jaccard.iloc[0]:.3f} (n = {len(jac)})"
pd.DataFrame([row]).to_csv(OUT / 'XAI_L3_row.csv', index=False)
print(pd.DataFrame([row]).T)

## V6. 5 truy vấn thành công và 5 truy vấn thất bại (seed 2026)

Luật cố định, không nhìn giải thích: lấy `N_CASE_POOL` truy vấn test (bốc với `SELECTION_SEED`), tính hạng ảnh đúng của CLIP, mô hình
chính, `no_edge` và `random_edges`.
- **Thành công:** CLIP xếp ảnh đúng ở hạng > 1, CLIP + GAT xếp hạng 1, và cả `no_edge`, `random_edges` đều xếp hạng > 1
  (để thành công quy được cho cấu trúc quan hệ thật). Sắp theo mức tăng hạng, lấy 5.
- **Thất bại:** 3 ca "đồ thị làm hại" (CLIP hạng 1, CLIP + GAT hạng > 1) và 2 ca "không cứu được" (CLIP > 1, CLIP + GAT ở hạng 2 đến 5).
  Bốc ngẫu nhiên với `SELECTION_SEED`. Mỗi ảnh đúng chỉ xuất hiện một lần.

In [ ]:
pool = test_res.iloc[np.sort(np.random.default_rng(SELECTION_SEED + 1).choice(len(test_res), N_CASE_POOL, replace=False))]
rk = []
for row in pool.itertuples(index=False):
    ids, clip, gv, qz, fused, rank, crank, top = score_row('test', 'seed_2026', row)
    if str(row.positive_image_id) not in ids:
        continue
    r_no = score_row('test', 'no_edge', row)[5]
    r_rd = score_row('test', 'random_edges', row)[5]
    rk.append({'text_id': str(row.text_id), 'positive': str(row.positive_image_id), 'r_clip': crank, 'r_gat': rank,
               'r_no_edge': r_no, 'r_random': r_rd, 'top': ids[top]})
rk = pd.DataFrame(rk)
rk.to_csv(OUT / 'V6_case_pool_ranks.csv', index=False)
cnt = {'CLIP sai → GAT đúng': int(((rk.r_clip > 1) & (rk.r_gat == 1)).sum()),
       '  trong đó no_edge cũng đúng': int(((rk.r_clip > 1) & (rk.r_gat == 1) & (rk.r_no_edge == 1)).sum()),
       '  trong đó cần quan hệ thật': int(((rk.r_clip > 1) & (rk.r_gat == 1) & (rk.r_no_edge > 1) & (rk.r_random > 1)).sum()),
       'CLIP đúng → GAT sai': int(((rk.r_clip == 1) & (rk.r_gat > 1)).sum()), 'Số truy vấn': len(rk)}
print(json.dumps(cnt, ensure_ascii=False, indent=1))
(OUT / 'V6_rescue_counts.json').write_text(json.dumps(cnt, ensure_ascii=False, indent=1))

def pick(df, n, sort=None, seed=SELECTION_SEED):
    df = df.drop_duplicates('positive')
    df = df.sort_values(sort, ascending=False) if sort else df.sample(frac=1.0, random_state=seed)
    return df.head(n)
succ = pick(rk[(rk.r_clip > 1) & (rk.r_gat == 1) & (rk.r_no_edge > 1) & (rk.r_random > 1)].assign(gain=lambda d: d.r_clip - d.r_gat), 5, 'gain')
harm = pick(rk[(rk.r_clip == 1) & (rk.r_gat > 1) & ~rk.positive.isin(succ.positive)], 3)
miss = pick(rk[(rk.r_clip > 1) & (rk.r_gat.between(2, 5)) & ~rk.positive.isin(pd.concat([succ, harm]).positive)], 2)
CASES = ([('thanh_cong', r) for r in succ.itertuples(index=False)] + [('that_bai_lam_hai', r) for r in harm.itertuples(index=False)]
         + [('that_bai_khong_cuu', r) for r in miss.itertuples(index=False)])
print(len(CASES), 'ca')

## Hình giải thích (nhúng nguyên văn `vg_xai_plot.py`): ảnh | đồ thị giải thích | bảng quan hệ, chỉ vẽ đúng tập S

In [ ]:
# Hình giải thích cấp 1, cùng kiểu với hình ca cấp 2 (07 / 18.py):
# ảnh | đồ thị giải thích (spring layout, nét xanh dương = quan hệ trong S, độ dày và số trong ngoặc = Δ điểm đồ thị).
# Chỉ vẽ đúng tập giải thích S. Nút trùng tên được gộp (×n); quan hệ trùng (đầu, quan hệ, đuôi) cộng Δ.
import math
import textwrap
from collections import OrderedDict

import matplotlib.pyplot as plt
import networkx as nx

# Cùng bảng màu với các biểu đồ của báo cáo (report/figures/scripts/make_figures.py):
# xanh dương = tập giải thích S, cam = ảnh/thực thể được trả về, xanh ngọc = thực thể neo, xám = nút khác.
C_EDGE, C_IMG, C_ANCHOR, C_OTHER = "#2a78d6", "#eb6834", "#1baf7a", "#d9d8d4"
C_LINK, C_QUERY = "#9cc0ef", "#52514e"
ROOT = "ẢNH"
# Cỡ chữ (pt) cho hình rộng FIG_W inch; khi chèn bằng \textwidth (7.16 in) chữ còn khoảng 6 đến 8 pt.
FS = dict(sup=11, title=10, node=9, edge=8.5, tick=9, label=9)
FIG_W, FIG_H = 8.0, 3.6


def _endpoints(g, u):
    """Chiều gốc của quan hệ (đầu, đuôi) theo chú thích Visual Genome."""
    e = g.unit_edges[u][0]
    s, d = int(g.src[e]), int(g.dst[e])
    if bool(g.rev[e]):
        s, d = d, s
    return s, d


def _rel(g, u, rel_name):
    r = rel_name.get(g.unit_keys[u][2], str(g.unit_keys[u][2]))
    return "quan hệ lạ" if r == "UNK_REL" else r


def merged_explanation(g, S, deltas, node_label, rel_name):
    """Trả về list ((đầu, quan hệ, đuôi), n, tổng Δ) sắp theo Δ giảm dần; đầu/đuôi là nhãn, ảnh gốc là ROOT."""
    key_of = lambda v: ROOT if v == g.root else node_label(g, v)
    groups = OrderedDict()
    for u in S:
        s, d = _endpoints(g, u)
        k = (key_of(s), _rel(g, u, rel_name), key_of(d))
        n, t = groups.get(k, (0, 0.0))
        groups[k] = (n + 1, t + float(deltas[u]))
    return sorted(((k, n, t) for k, (n, t) in groups.items()), key=lambda x: -x[2])


def _sentence(k):
    s, r, d = k
    return f"ảnh chứa {d}" if s == ROOT and r == "contains" else f"{s} {r} {d}"


def _layout(G, tries=30):
    """Spring layout (như cấp 2), chọn seed để không cạnh nào đi xuyên qua nút khác và các nút không quá sát nhau."""
    import numpy as np

    def penalty(pos):
        P = {k: np.asarray(v, float) for k, v in pos.items()}
        span = max(np.ptp([p[0] for p in P.values()]), np.ptp([p[1] for p in P.values()]), 1e-9)
        bad = 0.0
        for a, b in G.edges:
            A, B = P[a], P[b]
            AB = B - A
            for c, C in P.items():
                if c in (a, b):
                    continue
                t = np.clip(np.dot(C - A, AB) / max(np.dot(AB, AB), 1e-12), 0, 1)
                dist = np.linalg.norm(A + t * AB - C) / span
                bad += max(0.0, 0.12 - dist) * 10
        keys = list(P)
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                bad += max(0.0, 0.15 - np.linalg.norm(P[keys[i]] - P[keys[j]]) / span) * 5
        return bad

    best, best_pen = None, float("inf")
    for seed in range(tries):
        pos = nx.spring_layout(G, seed=seed, k=1.2)
        pen = penalty(pos)
        if pen < best_pen:
            best, best_pen = pos, pen
        if pen == 0:
            break
    return best


def draw_graph(ax, items, k_star):
    ax.axis("off")
    if not items:
        ax.text(.5, .5, "Không có quan hệ nào\nlàm tăng điểm đồ thị", ha="center", va="center", fontsize=FS["title"],
                transform=ax.transAxes)
        return
    G = nx.DiGraph()
    G.add_node(ROOT)
    size, mult, loops = {}, {}, {}
    # Δ ghi trên cạnh theo một thang 10^expo chung; độ dày cạnh tỉ lệ với Δ.
    expo = int(math.floor(math.log10(max(t for _, _, t in items))))
    for (s, r, d), n, t in items:
        for x in (s, d):
            G.add_node(x)
            if x != ROOT:
                size[x] = size.get(x, 0.0) + t
        if s == ROOT:
            mult[d] = max(mult.get(d, 1), n)
        lab = (("chứa" if r == "contains" else r) + (f" ×{n}" if n > 1 and s != ROOT else "")
               + f" ({t / 10 ** expo:.3g})")
        if s == d:                # hai vật cùng tên có quan hệ với nhau: ghi ngay dưới tên nút
            loops.setdefault(s, []).append(lab)
            continue
        if G.has_edge(s, d):      # nhiều quan hệ trên cùng một cặp: ghi đủ, không để nhãn sau đè nhãn trước
            G.edges[s, d]["label"] += " / " + lab
            G.edges[s, d]["delta"] += t
        else:
            G.add_edge(s, d, label=lab, delta=t)
    for e in G.edges:
        G.edges[e]["label"] = textwrap.shorten(G.edges[e]["label"], 40, placeholder="...")
    vmax = max(size.values()) if size else 1.0
    dmax = max(G.edges[e]["delta"] for e in G.edges) if G.edges else 1.0
    pos = _layout(G)
    nodes = list(G)
    nsize = {x: 900 if x == ROOT else 250 + 900 * max(size.get(x, 0), 0) / vmax for x in nodes}
    nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=nodes, node_color=[C_IMG if x == ROOT else C_OTHER for x in nodes],
                           node_size=[nsize[x] for x in nodes])
    # cặp có quan hệ theo cả hai chiều: uốn cong để hai cạnh và hai nhãn tách nhau
    two_way = [e for e in G.edges if G.has_edge(e[1], e[0])]
    one_way = [e for e in G.edges if e not in two_way]
    for elist, cs in ((one_way, "arc3"), (two_way, "arc3,rad=0.25")):
        if not elist:
            continue
        nx.draw_networkx_edges(G, pos, ax=ax, edgelist=elist, edge_color=C_EDGE, arrows=True, arrowsize=12,
                               width=[1.0 + 3.5 * G.edges[e]["delta"] / dmax for e in elist],
                               nodelist=nodes, node_size=[nsize[x] for x in nodes], connectionstyle=cs)
        nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels={e: G.edges[e]["label"] for e in elist},
                                     font_size=FS["edge"], connectionstyle=cs,
                                     bbox=dict(fc="white", ec="none", alpha=0.85, pad=0.5))
    labels = {x: x if x == ROOT else textwrap.shorten(x, 22, placeholder="...") + (f" ×{mult[x]}" if mult.get(x, 1) > 1 else "")
              for x in nodes}
    for x, labs in loops.items():
        labels[x] += "\n(" + ", ".join(labs) + f" {textwrap.shorten(x, 14, placeholder='...')})"
    nx.draw_networkx_labels(G, pos, ax=ax, labels=labels, font_size=FS["node"])
    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 0.45, max(xs) + 0.45); ax.set_ylim(min(ys) - 0.22, max(ys) + 0.22)
    ax.set_title(f"Giải thích S (K* = {k_star}); số trong ngoặc: Δ điểm đồ thị ×10$^{{{expo}}}$",
                 fontsize=FS["title"])


def _show_image(ax, image, title):
    if image is not None:
        ax.imshow(image)
    else:
        ax.text(.5, .5, "không tải được ảnh", ha="center", va="center", transform=ax.transAxes)
    ax.set_title(title, fontsize=FS["title"])
    ax.axis("off")


def case_figure_items(images, items, k_star, suptitle):
    """Như case_figure nhưng nhận thẳng items của merged_explanation (dùng khi vẽ lại từ V6_cases.md)."""
    n = len(images)
    fig, axes = plt.subplots(1, n + 1, figsize=(FIG_W / 2.6 * (n + 1.6), FIG_H),
                             gridspec_kw={"width_ratios": [1.0] * n + [1.6]})
    for ax, (title, img) in zip(axes, images):
        _show_image(ax, img, title)
    draw_graph(axes[n], items, k_star)
    fig.suptitle(suptitle, fontsize=FS["sup"])
    fig.tight_layout()
    return fig


def case_figure(images, g, S, deltas, node_label, rel_name, k_star, suptitle):
    """images: list (tiêu đề, ảnh PIL hoặc None); giải thích (g, S, deltas) là của ảnh đầu tiên, như cấp 2."""
    return case_figure_items(images, merged_explanation(g, S, deltas, node_label, rel_name), k_star, suptitle)


def plot_explained_row(axes, g, S, deltas, image, header, node_label, rel_name, k_star):
    """Một hàng (ảnh, đồ thị) cho demo nhiều kết quả."""
    _show_image(axes[0], image, header)
    items = merged_explanation(g, S, deltas, node_label, rel_name)
    draw_graph(axes[1], items, k_star)


def new_figure(n_rows):
    fig, axes = plt.subplots(n_rows, 2, figsize=(FIG_W, FIG_H * n_rows), squeeze=False,
                             gridspec_kw={"width_ratios": [1.0, 1.6]})
    return fig, axes


def unit_sentence(g, u, node_label, rel_name):
    s, d = _endpoints(g, u)
    r = _rel(g, u, rel_name)
    if r == "contains":
        return f"ảnh chứa {node_label(g, d if s == g.root else s)}"
    return f"{node_label(g, s)} {r} {node_label(g, d)}"


def vg_image_urls(image_id):
    """Đường dẫn ảnh Visual Genome công khai theo số hiệu ảnh (hai thư mục của bộ dữ liệu gốc)."""
    num = str(image_id).split(":")[-1]
    return [f"https://cs.stanford.edu/people/rak248/VG_100K/{num}.jpg",
            f"https://cs.stanford.edu/people/rak248/VG_100K_2/{num}.jpg"]

In [ ]:
import requests, io
from PIL import Image

_IMG_CACHE = {}
def fetch_image(image_id):
    # Ưu tiên URL trong images.parquet, sau đó thử hai thư mục ảnh công khai của Visual Genome.
    key = str(image_id)
    if key in _IMG_CACHE:
        return _IMG_CACHE[key]
    urls = ([IMG_URL[key]] if IMG_URL.get(key) else []) + vg_image_urls(key)
    img = None
    for url in urls:
        try:
            r = requests.get(url, timeout=15); r.raise_for_status()
            img = Image.open(io.BytesIO(r.content)).convert('RGB'); break
        except Exception:
            continue
    _IMG_CACHE[key] = img
    return img

def explain_image(name, row, image_id):
    spec = MODELS[name]
    ids, clip, gv, qz, fused, rank, crank, top = score_row('test', name, row)
    i = ids.index(image_id)
    g = ComponentGraph(compact, image_to_component[image_id], image_features, entity_features, relation_type_features, DEVICE)
    s_full, s_none, deltas = unit_deltas(spec['model'], g, qz)
    S = explanation(deltas, K_STAR)
    s_rm = float(graph_scores(spec['model'], g, qz, g.keep_mask_without(S).unsqueeze(0))[0]) if S else s_full
    _, r_rm = prob_rank(clip, gv, spec['alpha'], i, s_rm)
    _, r0 = prob_rank(clip, gv, spec['alpha'], i)
    return g, S, deltas, {'S_graph': s_full, 'structure': s_full - s_none, 'rank': r0, 'rank_without_S': r_rm}

md_lines = [f'# Visual Genome: 5 truy vấn thành công và 5 truy vấn thất bại (seed 2026, K* = {K_STAR})', '']
for kind, c in CASES:
    row = test_res[test_res.text_id.astype(str) == c.text_id].iloc[0]
    text = SPLIT['test']['text'].get(c.text_id, '')
    targets = [('Ảnh đúng', c.positive)] + ([('Ảnh hạng 1 (sai)', c.top)] if c.top != c.positive else [])
    md_lines += [f'## {kind}: {c.text_id}', f'**Truy vấn:** {text}  ',
                 f'**Hạng ảnh đúng:** CLIP {c.r_clip} → CLIP + GAT {c.r_gat} · no_edge {c.r_no_edge} · random_edges {c.r_random}  ',
                 f'**Hình:** `cases/{kind}_{c.text_id}.png`', '']
    panels, first = [], None
    for lab, img_id in targets:
        g, S, deltas, info = explain_image('seed_2026', row, img_id)
        first = first or (g, S, deltas)
        panels.append((f"{lab}\nhạng {info['rank']}; bỏ giải thích S: hạng {info['rank_without_S']}", fetch_image(img_id)))
        md_lines += [f'**{lab}** ({img_id}): điểm đồ thị {info["S_graph"]:.4f}, phần do quan hệ {info["structure"]:+.4f}, '
                     f'hạng {info["rank"]}, bỏ giải thích → hạng {info["rank_without_S"]}', '',
                     '| # | Quan hệ | Δ điểm đồ thị |', '|---|---|---:|']
        md_lines += [f'| {j+1} | {unit_sentence(g, u, node_label, REL_NAME)} | {float(deltas[u]):+.2e} |' for j, u in enumerate(S)] + ['']
    # như cấp 2: đồ thị và cột Δ là giải thích của ảnh đúng
    fig = case_figure(panels, *first, node_label, REL_NAME, K_STAR,
                      f"[{kind}] {text}\nCLIP hạng {c.r_clip} → CLIP + GAT hạng {c.r_gat}")
    fig.savefig(OUT / 'cases' / f'{kind}_{c.text_id}.png', dpi=200, bbox_inches='tight'); plt.close(fig)
(OUT / 'V6_cases.md').write_text('\n'.join(md_lines), encoding='utf-8')
print('\n'.join(md_lines[:40]))

## V7. Demo: nhập câu truy vấn → top-k ảnh + đồ thị giải thích

Câu tự do được mã hóa bằng CLIP ViT-B/32 (OpenAI) đóng băng; ô đầu kiểm tra encoder khớp embedding câu đã trích của nhóm
(cosine ≥ 0,99) trước khi dùng. Ứng viên là 10.443 ảnh test; CLIP lấy top-50, mô hình seed 2026 xếp hạng lại với α đã khóa.

In [ ]:
import open_clip
clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai', device=DEVICE)
tokenizer = open_clip.get_tokenizer('ViT-B-32'); clip_model.eval()
@torch.no_grad()
def encode_text(texts):
    return F.normalize(clip_model.encode_text(tokenizer(texts).to(DEVICE)).float(), dim=1).cpu()
chk = test_res.head(20)
mine = encode_text([SPLIT['test']['text'][str(t)] for t in chk.text_id])
theirs = F.normalize(torch.stack([SPLIT['test']['tfeat'][SPLIT['test']['trow'][str(t)]].float() for t in chk.text_id]), dim=1)
cos = (mine * theirs).sum(1)
print('cosine encoder demo so với embedding của nhóm: min', float(cos.min()))
DEMO_OK = float(cos.min()) >= 0.99
img_emb = torch.load(CLIP_ROOT / 'test/image_embeddings.pt', map_location='cpu', weights_only=True).float()
img_idx = pd.read_parquet(CLIP_ROOT / 'test/image_embedding_index.parquet')
img_ids = img_idx.sort_values('embedding_row').image_id.astype(str).tolist() if 'embedding_row' in img_idx else img_idx.image_id.astype(str).tolist()

@torch.no_grad()
def demo(text, k=3, name='seed_2026'):
    assert DEMO_OK, 'Encoder demo không khớp embedding của nhóm: không dùng demo để báo cáo'
    spec = MODELS[name]
    q = encode_text([text])[0]
    clip_all = F.normalize(img_emb, dim=1) @ q
    top50 = torch.topk(clip_all, 50).indices.tolist()
    ids = [img_ids[j] for j in top50]
    qz = spec['model'].encode_query(q.unsqueeze(0).to(DEVICE))[0]
    gv = (split_emb('test', name)[[SPLIT['test']['row'][x] for x in ids]] @ qz.cpu()).float()
    fused = fuse(clip_all[top50], gv, spec['alpha'])
    order = torch.argsort(fused, descending=True).tolist()[:k]
    fig, axes = new_figure(k)
    for c, j in enumerate(order):
        g = ComponentGraph(compact, image_to_component[ids[j]], image_features, entity_features, relation_type_features, DEVICE)
        _, _, deltas = unit_deltas(spec['model'], g, qz)
        S = explanation(deltas, K_STAR)
        header = (f"#{c+1} (CLIP hạng {j+1}), điểm cuối {float(fused[j]):+.2f}\n"
                  f"cosine CLIP {float(clip_all[top50[j]]):.3f}, điểm đồ thị {float(gv[j]):.3f}")
        plot_explained_row(axes[c], g, S, deltas, fetch_image(ids[j]), header, node_label, REL_NAME, K_STAR)
    fig.suptitle(f'Truy vấn: "{text}"  (α_CLIP = {spec["alpha"]}, K* = {K_STAR})', fontsize=9)
    fig.tight_layout()
    return fig

for q in ['a man riding a horse on the beach', 'a cat sleeping on a laptop']:
    f = demo(q); f.savefig(OUT / f"demo_{q.replace(' ', '_')[:40]}.png", dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
info = {'notebook': '11_visualgenome_gat_xai_level1', 'K_star': K_STAR, 'seeds': SEEDS, 'main_seed': MAIN_SEED,
        'alpha_clip': {k: v['alpha'] for k, v in MODELS.items()}, 'checkpoint_sha256': {k: v['sha256'] for k, v in MODELS.items()},
        'n_val_K': N_VAL_K, 'n_test': N_TEST, 'n_random': N_RANDOM, 'K_grid': K_GRID, 'n_case_pool': N_CASE_POOL,
        'selection_seed': SELECTION_SEED, 'entity_label_column': ENT_LABEL_COL, 'finished': time.strftime('%Y-%m-%d %H:%M:%S')}
(OUT / '11_run_info.json').write_text(json.dumps(info, indent=2, ensure_ascii=False)); print(info)